# Text classification and error analysis

A customs officer who receives a new request wants a suggestion: which heading is this product most likely in? This page sets up that task, the EBTI case study of Sessions 13–16, and builds the first model for it on the TF-IDF representation of [Block 1](01-lecture-text-as-data.ipynb). Linear models are the standard classifiers for this kind of sparse, high-dimensional input, even with more than 1,000 classes. With so many classes, most of them rare, a single accuracy figure is not enough, so the page shows how to evaluate such a model: accuracy and macro-F1, per-heading results by frequency, and validation by time instead of a random split.

A score tells us how often a model fails, not why. The second half of the page analyses the errors (a confusion matrix for neighbouring headings, misclassified decisions read next to their English keywords and heading texts, the n-grams the model relies on), measures a leak that error analysis can hide (the customs' justification, a text that exists only after the decision), introduces truncated SVD as a way to compress the sparse matrix, and ends with the limits of word counts that motivate the language models of Session 14.

The code cells in this notebook build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start. The whole notebook runs in about seven minutes on a laptop; the character n-gram model takes about two of them.

```mermaid
flowchart LR
    T["Training decisions<br/>2017-2021"] --> V["TfidfVectorizer<br/>(words or char_wb 3-5)"]
    V --> C["Linear SVM<br/>(SGDClassifier, hinge)"]
    C --> E["Validation 2022-2023:<br/>accuracy, macro-F1,<br/>per-heading report"]
    E --> A["Error analysis:<br/>confusions, 20 errors,<br/>top n-grams"]
    A -->|"change features<br/>or data"| V
    E -->|"choose settings"| F["Refit on all<br/>training decisions"]
    F --> S["Leaderboard L1<br/>(workbook 04)"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## The EBTI task and the leaderboard

### Concept

**The decision to support.** A trader asks a customs authority how a product is classified in the EU tariff; customs answers with a Binding Tariff Information (BTI) decision that states the code, and the European Commission publishes all decisions in the EBTI database. Classifying a product takes an expert's time, and the tariff has more than 1,000 four-digit **headings**. A model that reads the description of goods and proposes the most likely headings can save that time, provided we know how often it is right and where it fails.

**The data** ([case-study/README.md](../../../case-study/README.md)):

| Table | Rows | What it contains |
|---|---|---|
| `train` | 309,529 decisions, 2017–2023 | description, language, issuing country, start date, the label `heading`, and columns written by customs together with the decision (keywords, justification, full code) |
| `train_sample` | 50,000 | a random sample of `train`, used for all experiments in this session |
| `test` | 113,188 decisions, 2024–2026 | only what a request contains: `id`, issuing country, language, start date, description |
| `nomenclature` | 1,229 headings | the English text of every heading, chapter and section |

Descriptions are written in the language of the issuing country (57 % German, 16 % French, 5 % English, 23 languages in all); every training decision also has English keywords that help us read it.

**The split is by time.** The model is trained on decisions up to 2023 and predicts later ones, as it would in use. The **public leaderboard** scores the decisions of 2024, the **private leaderboard** those of 2025–2026, which stay hidden until the end of the course. Test decisions whose description repeats a training description are removed, so the task cannot be solved by looking up old decisions. A submission is a CSV file with one row per test decision and the columns `id,heading`. The metric is **accuracy**, with **macro-F1** reported alongside (both explained below). There are three rounds: **L1** with a TF-IDF classifier (this session), **L2** with embeddings or a language model (Session 14), and **L3** after retraining with the released 2024 labels (Session 16). The leaderboard is not graded, because decisions can be looked up in the public database.

### Why it matters

Before choosing a method we fix what a good answer is. The test set contains only what a request contains, so a model may use only those columns; the time split makes the validation score a forecast of performance on future requests; and the leaderboard gives every team the same hidden test, so results can be compared honestly across the three rounds.

### How it works in Python

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

decisions = pd.read_parquet("case-study/data/train_sample.parquet")
test = pd.read_parquet("case-study/data/test.parquet")
print(len(decisions), decisions["heading"].nunique(), len(test))   # 50000 934 113188
print(test.columns.tolist())   # ['id', 'issuing_country', 'language', 'start_date', 'description']
print(test["start_date"].dt.year.value_counts().sort_index().to_dict())
# {2024: 40369, 2025: 41126, 2026: 31693}: 2024 public, 2025-2026 private
print(decisions["language"].value_counts(normalize=True).head(3).round(2).to_dict())
# {'de': 0.57, 'fr': 0.16, 'en': 0.05}
print(decisions["heading"].value_counts().head(3).to_dict())       # {'3926': 2009, '9503': 1424, '6307': 1374}

50000 934 113188
['id', 'issuing_country', 'language', 'start_date', 'description']
{2024: 40369, 2025: 41126, 2026: 31693}
{'de': 0.57, 'fr': 0.16, 'en': 0.05}
{'3926': 2009, '9503': 1424, '6307': 1374}


### In practice

- Customs administrations and the World Customs Organization have built tools that suggest HS codes from the free-text goods descriptions of declarations; an officer confirms or corrects the suggestion.
- Kaggle and Codabench competitions follow the protocol of the course leaderboard: a hidden test set, a fixed submission format, a public leaderboard during the competition and a private one for the final ranking.

> **Warning.** `keywords`, `classification_justification`, `cn_code`, `chapter`, `status` and the end dates are written by customs together with or after the decision. They are not in the test set and must not be model inputs. The leakage section below measures what happens when this rule is broken.

## Linear models for text classification

### Concept

A **linear classifier** computes one score per class as a weighted sum of the features, plus an intercept: score_k(d) = b_k + Σ_j w_kj x_dj. It predicts the class with the highest score. For text, each feature is one n-gram, so each weight says how much that n-gram pushes a decision towards a heading.

Worked example: suppose the model has the weights below for heading 6404 (textile uppers) and a description has TF-IDF values `textile` = 0.5, `leather` = 0.4, `textile upper` = 0.6.

| n-gram | weight (6404) | value | product |
|---|---|---|---|
| textile | 2.0 | 0.5 | 1.0 |
| leather | −3.0 | 0.4 | −1.2 |
| textile upper | 4.0 | 0.6 | 2.4 |

With intercept −1.0 the score for 6404 is −1.0 + 1.0 − 1.2 + 2.4 = 1.2. The bigram outweighs the word "leather" (which here refers to the sole). The model computes such a score for each of the 890 headings in the training data and picks the largest.

Three linear classifiers are common for text:

| Model | Idea | Output | Notes |
|---|---|---|---|
| Multinomial or complement naive Bayes | word probabilities per class, assumed independent | probabilities (poorly calibrated) | very fast; works on counts |
| Logistic regression | minimises log-loss; softmax turns scores into probabilities (Session 6) | probabilities | slow with 1,000 classes and 300,000 features |
| Linear support vector machine | maximises the margin between classes | scores only | here trained with `SGDClassifier(loss="hinge")` |

With many classes, scikit-learn trains one binary model per class ("one versus rest"). `SGDClassifier` fits these models by **stochastic gradient descent**: it passes over the data a fixed number of times (`max_iter`) and updates the weights after each example, which is fast enough for 300,000 decisions and 1,100 headings. Its parameter `alpha` is the strength of the **regularisation** penalty (larger = stronger); `LogisticRegression` and `LinearSVC` use `C`, the inverse.

The vectoriser and the classifier belong into one **pipeline**. Then `fit` learns the vocabulary and idf weights only from the training part.

**Validation by time.** The leaderboard asks for decisions of 2024–2026 given decisions of 2017–2023, so we validate the same way: train on 2017–2021, validate on 2022–2023. The section on validation below measures how much a random split would overstate the result.

### Why it matters

Linear models are fast on sparse input, need little tuning, and are easy to inspect, because each weight belongs to a readable n-gram. A TF-IDF linear model is the baseline every more complex text model has to beat; on the full training set it reaches 0.87–0.88 public accuracy (case-study README). Tree ensembles (Session 10) are a poor fit here: they split on one feature at a time and struggle with hundreds of thousands of sparse columns and 1,000 classes.

### How it works in Python

In [3]:
import time

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline

year = decisions["start_date"].dt.year
train, valid = decisions[year <= 2021], decisions[year >= 2022]      # validation by time
print(len(train), len(valid), train["heading"].nunique())            # 36801 13199 890


def svm():                                       # a linear SVM fitted by stochastic gradient descent
    return SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0, n_jobs=-1)


models = {
    "naive Bayes, word counts 1-2": make_pipeline(CountVectorizer(ngram_range=(1, 2), min_df=2),
                                                  ComplementNB()),
    "linear SVM, word TF-IDF 1": make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), svm()),
    "linear SVM, word TF-IDF 1-2": make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), svm()),
}
for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(train["description"], train["heading"])
    pred = model.predict(valid["description"])
    print(f"{name:30s} acc {accuracy_score(valid['heading'], pred):.3f}  "
          f"macro-F1 {f1_score(valid['heading'], pred, average='macro'):.3f}  "
          f"{time.perf_counter() - t0:.0f} s")
# naive Bayes, word counts 1-2   acc 0.673  macro-F1 0.392  26 s
# linear SVM, word TF-IDF 1      acc 0.772  macro-F1 0.511  24 s
# linear SVM, word TF-IDF 1-2    acc 0.769  macro-F1 0.507  37 s

# character n-grams inside word boundaries (about two minutes)
char = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                     sublinear_tf=True, max_features=300_000), svm())
char.fit(train["description"], train["heading"])
pred_char = char.predict(valid["description"])
print(round(accuracy_score(valid["heading"], pred_char), 3),
      round(f1_score(valid["heading"], pred_char, average="macro"), 3))   # 0.788 0.529

36801 13199 890


naive Bayes, word counts 1-2   acc 0.673  macro-F1 0.392  25 s


linear SVM, word TF-IDF 1      acc 0.772  macro-F1 0.511  14 s


linear SVM, word TF-IDF 1-2    acc 0.769  macro-F1 0.507  31 s


0.788 0.529


Results of further runs (same split, documented in the course team's experiments): no lower-casing 0.758; `LinearSVC` on word unigrams 0.769 in 96 seconds.

### In practice

- Routing e-mails and support tickets to the right team is often done with TF-IDF and a linear model, because it is cheap to retrain when categories change.
- Automatic coding of occupations, industries and products into official classifications (ISCO, NACE, HS) is a long-standing application of linear text classifiers in official statistics, often with a human checking low-confidence cases (Session 14, abstention).
- Joulin et al. (2017) showed with fastText that a linear classifier on word and character n-grams reaches accuracy close to deep neural networks on standard text-classification benchmarks while training in seconds.

> **Warning.** **Do not compare models on the test set.** Choose the analyzer, n-gram range, `min_df` and `alpha` on the validation years. The leaderboard is used once per round, for the final choice.

> **Tip.** Grid search over a text pipeline multiplies the fitting time. With 1,000 classes, start with one setting per idea (word vs character, unigrams vs bigrams), keep the best, and only then tune `alpha` on a small grid.

> 📖 **Read more:** [extras/03-sklearn-text-classification-sparse-features.ipynb](extras/03-sklearn-text-classification-sparse-features.ipynb) (scikit-learn gallery): a linear classifier on TF-IDF features of the 20 Newsgroups posts with its confusion matrix, then a benchmark of eight classifiers comparing accuracy with training and prediction time.

## Accuracy and macro-F1 with more than 1,000 classes

### Concept

With two classes, Session 8 computed precision, recall and F1 for the positive class. With many classes the same measures are computed **per class**, treating that class as positive and all others as negative (one-vs-rest):

- **Precision** of heading k = correct predictions of k / all predictions of k. "When the model says 9503 (toys), how often is it right?"
- **Recall** of heading k = correct predictions of k / all true members of k. "Of all toy decisions, how many does the model find?"
- **F1** = 2 × precision × recall / (precision + recall), the harmonic mean; it is low if either of the two is low.
- **Support** = the number of true members of k in the evaluation data.

Three averages summarise the per-class values:

| Average | Definition | Effect with rare classes |
|---|---|---|
| macro | plain mean of the per-class F1 values | each of the ~800 headings counts equally, also those with one decision |
| weighted | mean weighted by support | dominated by frequent headings |
| micro | computed from all predictions pooled; equals accuracy for single-label tasks | dominated by frequent headings |

Worked example with ten decisions: six of heading 3926 (plastic articles), two of 6403 (leather footwear) and two of 6404 (textile footwear). The model gets all six 3926 right, one of two 6403 and one of two 6404; the two errors are predicted as 3926. Per-class F1: 3926 0.86 (precision 6/8, recall 6/6), 6403 0.67, 6404 0.67. Macro-F1 = (0.86 + 0.67 + 0.67) / 3 = 0.73; accuracy = 8 / 10 = 0.80.

### Why it matters

The headings have a **long tail**: the most frequent heading (3926) covers 4 % of the decisions, and 466 of the 890 headings in the training years 2017–2021 of the sample have fewer than ten decisions. Accuracy is dominated by the few hundred frequent headings and answers "what share of requests gets the right suggestion?". Macro-F1 gives a heading with 3 decisions the same weight as one with 2,000 and answers "how well does the model know the tariff as a whole?". Always predicting 3926 gives accuracy 0.04 and a macro-F1 close to 0. A model that is good on frequent headings and poor on rare ones reaches a high accuracy and a modest macro-F1, which is what the reference models of the leaderboard show (word TF-IDF on the full training set: accuracy 0.872, macro-F1 0.682 on 2024). The leaderboard ranks by accuracy, because an officer's workload depends on the share of requests; macro-F1 is reported next to it so that the tail stays visible.

### How it works in Python

In [4]:
y_true = ["3926"] * 6 + ["6403", "6403", "6404", "6404"]
y_pred = ["3926"] * 6 + ["3926", "6403", "3926", "6404"]
print(f1_score(y_true, y_pred, average=None, labels=["3926", "6403", "6404"]).round(2))   # [0.86 0.67 0.67]
print(round(f1_score(y_true, y_pred, average="macro"), 2), accuracy_score(y_true, y_pred))   # 0.73 0.8

always_3926 = np.repeat("3926", len(valid))                        # the simplest baseline
print(round(accuracy_score(valid["heading"], always_3926), 3),
      round(f1_score(valid["heading"], always_3926, average="macro"), 4))   # 0.04 0.0001

pred = models["linear SVM, word TF-IDF 1"].predict(valid["description"])
for avg in ["macro", "weighted", "micro"]:
    print(avg, round(f1_score(valid["heading"], pred, average=avg, zero_division=0), 3))
# macro 0.511 / weighted 0.76 / micro 0.772 (= accuracy)

[0.86 0.67 0.67]
0.73 0.8
0.04 0.0001


macro 0.511
weighted 0.76
micro 0.772


### In practice

- Shared tasks in text classification, such as SemEval-2017 Task 4 (Rosenthal et al., 2017), use macro-averaged measures so that rare classes count.
- Automatic coding of occupations, causes of death or economic activities by statistical offices deals with hundreds of codes and a long tail; evaluations report accuracy together with per-class or macro-averaged results.

> **Warning.** `f1_score` on more than two classes needs `average=`; always say which average you report. Macro-F1 also depends on which classes appear in the evaluation set at all, so compare macro-F1 values only on the same set.

## Per-heading results and rare headings

### Concept

A list of 1,000 per-heading F1 values cannot be read, but two views can: the per-heading report sorted by support, and the mean F1 of headings **grouped by how many training decisions they have**. The second view answers directly whether the errors concentrate in the tail.

Worked example: the validation years contain 523 decisions of heading 3926 (other articles of plastics). The model predicts 3926 for 655 decisions, of which 442 are correct. Precision = 442 / 655 = 0.67, recall = 442 / 523 = 0.85, F1 = 2 × 0.67 × 0.85 / 1.52 = 0.75. The model uses 3926 as a fallback: many decisions about other plastic goods (3923 containers, 3924 tableware) are pushed into it, which lowers its precision.

### Why it matters

In the validation years, 415 of the 778 headings that occur have fewer than five decisions, and 237 headings get F1 = 0. The grouped table below shows why: F1 depends strongly on how many training examples a heading has. This tells us what to improve: more data for rare headings helps more than tuning the frequent ones.

### How it works in Python

In [5]:
from sklearn.metrics import classification_report

report = pd.DataFrame(classification_report(valid["heading"], pred, output_dict=True,
                                            zero_division=0)).T.iloc[:-3]   # drop the averages
print(report.sort_values("support", ascending=False).head(5).round(2))
#       precision  recall  f1-score  support
# 3926       0.67    0.85      0.75    523.0
# 9503       0.77    0.96      0.85    372.0
# 6307       0.74    0.90      0.81    360.0
# 2106       0.73    0.90      0.81    347.0
# 4202       0.89    0.95      0.92    325.0
print(len(report), int((report["f1-score"] == 0).sum()), int((report["support"] < 5).sum()))
# 778 237 415: headings in the validation years, with F1 = 0, with fewer than 5 decisions

# F1 by the number of training decisions of a heading
n_train = train["heading"].value_counts().reindex(report.index, fill_value=0)
bins = pd.cut(n_train, [-1, 0, 10, 50, 200, np.inf], labels=["0", "1-10", "11-50", "51-200", ">200"])
print(report["f1-score"].groupby(bins, observed=True).agg(["mean", "count"]).round(2))
#         mean  count
# 0       0.00     44      <- headings never seen in training cannot be predicted
# 1-10    0.36    328
# 11-50   0.64    250
# 51-200  0.77    120
# >200    0.79     36

      precision  recall  f1-score  support
3926       0.67    0.85      0.75    523.0
9503       0.77    0.96      0.85    372.0
6307       0.74    0.90      0.81    360.0
2106       0.73    0.90      0.81    347.0
4202       0.89    0.95      0.92    325.0
778 237 415
        mean  count
count              
0       0.00     44
1-10    0.36    328
11-50   0.64    250
51-200  0.77    120
>200    0.79     36


### In practice

- Content-moderation systems report recall per category (for example spam, hate speech, self-harm), because a high overall accuracy can hide a category that is almost never detected.
- Medical screening tests are described by sensitivity (recall of the "ill" class) and specificity (recall of the "healthy" class), the same per-class view under different names.
- Extreme multi-label classification, for example assigning codes from large medical or product taxonomies, reports macro-averaged and "tail" metrics separately, because a model can look good on frequent codes while ignoring thousands of rare ones.

> **Important.** A model trained on 50,000 decisions sees on average about 55 examples per heading, but the median heading has far fewer. More training data (the full 309,529 decisions) helps the tail most: on the public leaderboard, macro-F1 rises from 0.52 to 0.68 while accuracy rises from 0.81 to 0.87 (case-study README).

> **Note.** An officer can check three suggestions in seconds, so a model whose second or third suggestion is often right is more useful than its accuracy says. **Top-k accuracy** counts a prediction as correct if the true heading is among the k highest-scoring headings; [workbook 04](04-case-study-tfidf-leaderboard.ipynb), section 2, computes it for the case study.

## Validation by time versus a random split

### Concept

A **random split** draws the validation decisions from all years; a **split by time** validates on the most recent years only. Both use the same number of decisions, but they answer different questions. A random split asks "how well does the model classify a decision like the ones it was trained on?"; a split by time asks "how well will it classify next year's requests?", which is the leaderboard's question and the officer's.

### Why it matters

Three things make the random split too optimistic here. Renewed decisions with near-identical descriptions from the same trader end up on both sides; the product mix and the wording change over the years (Session 16 measures this drift); and headings that were changed in the HS 2022 revision appear in training with their new meaning only in the later years. On the sample, the random split reports 0.816 accuracy and 0.571 macro-F1, the split by time 0.772 and 0.511. The leaderboard sides with the time split: the same model, trained on all 50,000 decisions, scores 0.803 on 2024 and 0.776 on 2025–2026 ([workbook 04](04-case-study-tfidf-leaderboard.ipynb) writes such a submission).

### How it works in Python

In [6]:
from sklearn.model_selection import train_test_split

rand_train, rand_valid = train_test_split(decisions, test_size=len(valid), random_state=0)
random_model = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), svm())
random_model.fit(rand_train["description"], rand_train["heading"])
pred_r = random_model.predict(rand_valid["description"])
print(round(accuracy_score(rand_valid["heading"], pred_r), 3),
      round(f1_score(rand_valid["heading"], pred_r, average="macro"), 3))   # 0.816 0.571
# time split with the same sizes (36,801 / 13,199): 0.772 0.511
print(round(rand_valid["description"].isin(rand_train["description"]).mean(), 3),
      round(valid["description"].isin(train["description"]).mean(), 3))   # 0.016 0.004: exact repeats

0.816 0.571


0.016 0.004


Exact repeats explain only a small part of the gap (1.6 % against 0.4 % of the validation descriptions); most of it comes from near-repeats and from change over time.

### In practice

- Forecasting competitions and credit-scoring teams validate "out of time": the model is fitted on older periods and tested on the most recent one, because that is how it will be used.
- Kapoor and Narayanan (2023) list random splits of temporal data among the common forms of leakage in published machine-learning studies.

> **Caution.** Choose models and settings on the validation years, never on the leaderboard. Every submission chosen by its public score is a small step of tuning on the test set; use the leaderboard to check, not to search.

## Error analysis of text models

```mermaid
flowchart LR
    A["Validation<br/>predictions"] --> B["Where are the errors?<br/>(confusion, top pairs)"]
    B --> C["Read a sample of errors<br/>with keywords and<br/>heading texts"]
    C --> D["Sort into<br/>categories"]
    D --> E["Count per<br/>category"]
    E --> F{"Fixable with<br/>text features?"}
    F -->|yes| G["Change features<br/>or data, re-validate"]
    F -->|no| H["Accept, document,<br/>or route to a person"]
    G --> A
```

### Concept

A **confusion matrix** counts, for every true class (row), how often each class was predicted (column). The diagonal holds the correct predictions; every off-diagonal cell is one kind of error. Dividing each row by its total gives the **recall** of each class on the diagonal. With 1,000 headings the full matrix has a million cells and cannot be read; we look instead at **groups of neighbouring headings** (one chapter) and at the **most frequent pairs** of true and predicted heading.

**Error analysis** means reading a sample of misclassified examples and sorting them into categories. For a multilingual corpus two helpers make this possible for everyone: the English **keywords** that customs assigned to each training decision, and the English **heading description** from the nomenclature. Keywords are *not* a model input (customs assigns them together with the classification, and the test set does not have them), but they are allowed for reading.

Categories that occur in the case study:

| Category | Example from the validation years (shortened) |
|---|---|
| material versus function | a fibre-optic splice box "mainly made of plastic" (true 8536, switching apparatus; predicted 3926, articles of plastics) |
| borderline between related headings | curry powder for sauces (true 0910, spices; predicted 2103, sauces and condiments) |
| rare language, few examples | a Slovak description of birch-wood bed slats (true 4421, wooden articles; predicted 7606, aluminium sheets) |
| technical or chemical knowledge needed | disodium pyrophosphate (true 2835, phosphates; predicted 2918, carboxylic acids) |
| sets and composite goods | a travel set of bottles and spatulas (true 9605, travel sets; predicted 3301, essential oils) |
| new heading after a nomenclature change | a nicotine product without tobacco (true 2404, created in HS 2022; predicted 3824) |

The **count per category** shows which improvement is worth trying. "Material versus function" and "borderline" errors follow from the legal rules of the tariff (the General Interpretative Rules and the section and chapter notes), which no description-only model sees; "rare language" errors may shrink with more data or character n-grams.

Worked example from the footwear chapter (figure below): the validation years contain 104 decisions of heading 6404 (textile uppers). The model labels 88 correctly, 10 as 6405 (other footwear), 5 as 6403 (leather uppers) and 1 as 6402. Recall for 6404 is 88 / 104 = 0.85. Heading 6406 (parts of footwear) has a recall of only 0.43: parts are described with the same words as complete shoes.

![Confusion matrix of the six footwear headings 6401 to 6406 in the validation years; recall is 0.98 for 6403, 0.85 for 6404 and 0.43 for 6406 (parts of footwear)](../figures/confusion_matrix.png)

### Why it matters

A single score says how often the model fails, not why. Error analysis turns the score into a list of concrete next steps, shows where the labels depend on rules the model cannot see, and gives an honest upper bound: if a third of the errors need the legal notes of the tariff, a better text representation alone will not fix them. It also prevents a common waste of time, namely tuning hyperparameters when the real problem lies in the data or the task.

### How it works in Python

In [7]:
from sklearn.metrics import confusion_matrix

nomenclature = pd.read_parquet("case-study/data/nomenclature.parquet").set_index("heading")
model = models["linear SVM, word TF-IDF 1"]       # the word model of the section on linear models
pred = model.predict(valid["description"])

shoes = ["6401", "6402", "6403", "6404", "6405", "6406"]
in64 = valid["heading"].isin(shoes).to_numpy()
print(confusion_matrix(valid["heading"][in64], pred[in64], labels=shoes))
# [[  1   0   0   0   0   0]
#  [  0  44   2   0   0   0]
#  [  0   0 112   0   0   0]
#  [  0   1   5  88  10   0]
#  [  0   0   0   0  71   0]
#  [  0   1   0   0   3   6]]   rows = true, columns = predicted (outside predictions not shown)

val = valid.assign(pred=pred)
wrong = val[val["heading"] != val["pred"]]
print(len(wrong), round((wrong["heading"].str[:2] == wrong["pred"].str[:2]).mean(), 2))
# 3003 0.3: only 30 % of the errors stay inside the right chapter
print(wrong[["heading", "pred"]].value_counts().head(3))
# 8524 -> 8548  17    flat-panel display modules: heading 8524 was created in HS 2022
# 8539 -> 9405  13    LED lamps versus luminaires
# 8524 -> 8529  12

# the 20 decisions of workbook 05 (fixed seed, so everyone reads the same ones)
errors20 = wrong.sample(20, random_state=1)
for _, row in errors20.head(2).iterrows():
    print(row["heading"], "->", row["pred"], row["language"],
          "|", nomenclature.loc[row["heading"], "heading_description"][:50],
          "|", nomenclature.loc[row["pred"], "heading_description"][:50])
    print("   keywords:", row["keywords"][:80])
# 9605 -> 3301 nl | Travel sets; for personal toilet, sewing, shoe or c | Oils; essential (concretes, absolutes); concentrat
#    keywords: AEROPLANES,BOTTLES,EMPTY,FOR FILLING,OF PLASTICS,SPATULAS,TRANSPARENT,TRAVEL SETS
# 9027 -> 8543 fr | Instruments and apparatus; for physical or chemical | Electrical machines and apparatus; having individu
#    keywords: ACIDS,AS MODULE,CHROMOTOGRAPHS,FOR MEDICAL USE,FOR PHARMACEUTICALS,PUMPS,SCIENTIFIC AND

[[  1   0   0   0   0   0]
 [  0  44   2   0   0   0]
 [  0   0 112   0   0   0]
 [  0   1   5  88  10   0]
 [  0   0   0   0  71   0]
 [  0   1   0   0   3   6]]
3003 0.3
heading  pred
8524     8548    17
8539     9405    13
8524     8529    12
Name: count, dtype: int64
9605 -> 3301 nl | Travel sets; for personal toilet, sewing, shoe or  | Oils; essential (concretes, absolutes); concentrat
   keywords: AEROPLANES,BOTTLES,EMPTY,FOR FILLING,OF PLASTICS,SPATULAS,TRANSPARENT,TRAVEL SET
9027 -> 8543 fr | Instruments and apparatus; for physical or chemica | Electrical machines and apparatus; having individu
   keywords: ACIDS,AS MODULE,CHROMOTOGRAPHS,FOR MEDICAL USE,FOR PHARMACEUTICALS,PUMPS,SCIENTI


Sorting the 20 sampled errors by hand gave the following counts. Your own categories may differ; the point is to count.

| Category | Count |
|---|---|
| borderline between related headings (orange drink 2202/2106, curry 0910/2103, tags 4821/4911, ...) | 6 |
| rare language or few examples (Czech, Slovak, Bulgarian, Polish, Spanish descriptions) | 5 |
| material versus function (plastic splice box, plastic filter column, wooden dry toilet, glass ornament with LED) | 4 |
| technical or chemical knowledge needed | 2 |
| sets and composite goods | 1 |
| new heading after HS 2022 (2404, nicotine products) | 1 |
| no clear reason | 1 |

Eleven of twenty errors (borderline, material versus function, sets) depend on classification rules rather than on vocabulary. Five involve languages with few training decisions, which suggests more data or character n-grams. This is the pattern the model comparison above already showed: character n-grams help a little, the full training set helps more, and a large part of the errors remains.

### In practice

- Andrew Ng's *Machine Learning Yearning* (2018) recommends reading about 100 misclassified development-set examples by hand and counting error categories before deciding what to improve.
- Customs classification is reviewed by people: an officer checks a suggested code, and BTI decisions themselves are binding only after a customs expert has classified the goods. A model that reports its confidence can route uncertain cases to an expert (Session 14, abstention).
- Northcutt, Athalye and Mueller (2021) found label errors in the test sets of ten widely used benchmark datasets; confident errors of a model were their main tool for finding them. In the EBTI data, some "errors" may also be decisions that were later revoked (`invalidation_reason`).

> **Tip.** Fix the random seed when you sample errors (`random_state=1`), so that the whole team discusses the same decisions, and write the category of each one into a column. A spreadsheet of 20–100 categorised errors is one of the most useful artefacts of a text project.

## Leakage through the customs' justification

### Concept

**Target leakage** (Sessions 7 and 9, with the leaking revenue columns of the Airbnb price model) means that a feature contains information about the label that will not be available when the model is used. Every training decision has a column `classification_justification`, the text in which customs explain their classification. It is written *with* the decision, and it usually names the result:

> CLASSIFICATION HAS BEEN DETERMINED IN ACCORDANCE WITH THE FOLLOWING: GIR 1 HAS BEEN USED TO CLASSIFY THE PRODUCT BY THE TERMS OF HEADING 4303 - ARTICLES OF APPAREL, CLOTHING ACCESSORIES AND OTHER ARTICLES OF FURSKIN ...

In the sample, 70 % of the justifications contain the decision's own heading, and a rule that simply reads the first four-digit number of the justification gets 64 % of the headings right, without any model. A new request has no justification, and the test set does not contain the column. The same holds for `keywords` and `cn_code`.

### Why it matters

No validation scheme detects this kind of leak, because the justification is present in every validation decision as well. A model that uses it looks excellent in validation and fails in use. The only protection is to ask, for every column, *"would I have this value at the moment of prediction?"*, and to answer it from how the data are produced, not from the scores. For an officer, the consequence would be concrete: a tool that was validated at 96 % and is wrong for three requests in ten.

### How it works in Python

Append the justification to the description and validate as a careless notebook would, with a random split inside the training years. Then apply the model where it will be used: to the decisions of 2022–2023, which, like a new request, have only a description.

In [8]:
from sklearn.model_selection import train_test_split

just = decisions["classification_justification"].fillna("")
print(round(np.mean([h in j for h, j in zip(decisions["heading"], just)]), 2))   # 0.7
first_number = just.str.extract(r"\b(\d{4})\b", expand=False)
print(round((first_number == decisions["heading"]).mean(), 2))   # 0.64: the answer, without a model

with_just = decisions["description"] + " " + just
past = (year <= 2021).to_numpy()
tr, va = train_test_split(np.flatnonzero(past), test_size=0.2, random_state=0)   # within 2017-2021
later = np.flatnonzero(~past)                                                    # 2022-2023
leak_model = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True),
                           SGDClassifier(alpha=1e-5, random_state=0, n_jobs=-1))
for name, X in [("description only", decisions["description"]), ("+ justification", with_just)]:
    leak_model.fit(X.iloc[tr], decisions["heading"].iloc[tr])
    acc_val = accuracy_score(decisions["heading"].iloc[va], leak_model.predict(X.iloc[va]))
    acc_use = accuracy_score(decisions["heading"].iloc[later],
                             leak_model.predict(decisions["description"].iloc[later]))   # no justification
    print(f"{name:17s} validation {acc_val:.3f}   in use (2022-2023, description only) {acc_use:.3f}")
# description only  validation 0.804   in use (2022-2023, description only) 0.753
# + justification   validation 0.957   in use (2022-2023, description only) 0.707

0.7
0.64


description only  validation 0.804   in use (2022-2023, description only) 0.753


+ justification   validation 0.957   in use (2022-2023, description only) 0.707


Validation reports a jump from 0.80 to 0.96. In use, where only the description exists, the leaking model is *worse* than the honest one (0.707 against 0.753): it has learned to rely on a text that is missing, and it learned less from the description itself.

### In practice

- KDD Cup 2008 (breast cancer detection): patient identifiers were predictive of the label because of how the data had been assembled; Kaufman et al. (2012) use it as a textbook case of leakage.
- Hospital data: a feature such as "antibiotic prescribed" can reveal the diagnosis a model is supposed to predict, because it is recorded after the doctor suspected it.
- Churn data: fields such as "reason for leaving" are filled only for customers who have already left.

> **Warning.** A large jump in the validation score from one new column is a warning sign, not a success. Check when and by whom that column is written before you celebrate. For the leaderboard: never use `classification_justification`, `keywords`, `cn_code`, `chapter`, `status`, `end_date` or `invalidation_reason` as inputs.

> **Note.** Reading the keywords and the justification **during error analysis** is fine, as in the previous section: they help a person understand a decision. The rule concerns model inputs only.

## The most informative n-grams per class

### Concept

In a linear model each feature has one **coefficient** per class. A large positive coefficient means that the n-gram pushes the score towards that class. Because TF-IDF features are on comparable scales (each row has length 1), sorting the coefficients gives a direct ranking of the most informative n-grams per class.

This inspection is also a check for **shortcuts**: features that predict the label for reasons unrelated to the content of the goods.

![Bar charts of the twelve words with the largest coefficients for headings 6403, 6404 and 9503; leather words in several languages and the number 6403 lead for 6403, toy words in French, German, Dutch, Swedish and English for 9503](../figures/top_ngrams.png)

The figure shows three things. First, the model learns the same concept separately in every language: `jouet`, `spielzeug`, `speelgoed`, `leksak` and `toy` for toys. Second, the material words that the legal text names (`cuir`, `leder`, `leather`; `textile`, `spinnstoffen`) dominate the footwear headings, as they should. Third, **numbers** are among the strongest features: `6403`, `1200`, `1910`, `1100`. Traders and customs often quote tariff codes in the description ("keine Ware der Position 6403", "sous-position 6404 19 10"). The case-study data replace numbers that repeat the decision's *own* code by `<CODE>`, but a quoted heading number survives when it comes with a different subheading.

### Why it matters

In the validation years, 24.6 % of the descriptions contain their own four-digit heading as a number. For these the model is right 98 % of the time; for the others only 70.5 %. Is this leakage? No: the numbers are part of what the trader or customs wrote, and the test descriptions contain them at a similar rate (26 % in 2024–2026), so the feature is available at prediction time. But it changes how we read the score: about a quarter of the task is "read the code that is already there", and the hard part is the remaining three quarters. Report both numbers.

### How it works in Python

In [9]:
import re

has_number = pd.Series([bool(re.search(rf"(?<!\d){h}(?!\d)", t))
                        for h, t in zip(valid["heading"], valid["description"])], index=valid.index)
print(round(has_number.mean(), 3))                         # 0.246 of descriptions quote their heading
print(val.groupby(has_number).apply(lambda g: round((g["heading"] == g["pred"]).mean(), 3)).to_dict())
# {False: 0.705, True: 0.98}

vec, clf = model.named_steps["tfidfvectorizer"], model.named_steps["sgdclassifier"]
names = vec.get_feature_names_out()
for heading in ["6403", "6404", "3926"]:
    k = list(clf.classes_).index(heading)
    print(heading, list(names[np.argsort(clf.coef_[k])[::-1][:8]]))
# 6403 ['cuir', '6403', '24', 'leder', '1200', 'leather', 'naturel', '24cm']
# 6404 ['kunststoff', 'textile', '1910', 'caoutchouc', '1100', 'sole', 'spinnstoffen', '6404']
# 3926 ['3926', 'ouvrage', 'plastique', 'statuetten', 'andere', 'műanyagból', 'bâche', 'kunststoffen']

0.246
{False: 0.705, True: 0.98}
6403 ['cuir', '6403', '24', 'leder', '1200', 'leather', 'naturel', '24cm']
6404 ['kunststoff', 'textile', '1910', 'caoutchouc', '1100', 'sole', 'spinnstoffen', '6404']
3926 ['3926', 'ouvrage', 'plastique', 'statuetten', 'andere', 'műanyagból', 'bâche', 'kunststoffen']


### In practice

- Ribeiro, Singh and Guestrin (2016) showed with their LIME method that a classifier for the 20 Newsgroups data separated "Christianity" from "atheism" mostly through e-mail header words such as "posting" and "host", not through the content.
- In medical imaging, Zech et al. (2018) found that pneumonia classifiers partly recognised the hospital that took the X-ray (from markers on the image) instead of the disease, a shortcut that failed in other hospitals.
- The EU's General Data Protection Regulation gives people subject to some automated decisions a right to meaningful information about the logic involved; for linear text models the coefficients are one direct way to provide it.

> **Warning.** Coefficients are comparable only when the features are on the same scale and not strongly correlated. `leder` and `leather` share their signal across languages, so neither coefficient alone tells the full story. Treat the ranking as a diagnostic, not as a causal statement about words.

> 📖 **Read more:** [extras/03-sklearn-text-classification-sparse-features.ipynb](extras/03-sklearn-text-classification-sparse-features.ipynb) (scikit-learn gallery): the top words per class on the same 20 Newsgroups data include e-mail header words; stripping headers, signatures and quotes lowers the score but gives an honest model, a leak like the justification column.

## Dimensionality reduction with truncated SVD

### Concept

The TF-IDF matrix has tens of thousands of sparse columns. **Truncated singular value decomposition** (truncated SVD) approximates it by k dense columns, for example k = 100. Each new column, a **component**, is a weighted combination of words that tend to occur together. Applied to a document-term matrix, the method is called **latent semantic analysis** (LSA; Deerwester et al., 1990).

The idea in one picture: the document-term matrix X (documents × words) is written approximately as a product X ≈ U Σ Vᵀ, where V (words × k) describes each component by its word weights and U Σ (documents × k) gives the coordinates of each document on the components. Words that co-occur load on the same component.

Truncated SVD is closely related to PCA (Session 11). The difference: PCA first subtracts the column means, which would turn every zero into a non-zero number and destroy sparsity. Truncated SVD works on the sparse matrix directly.

```mermaid
flowchart LR
    X["TF-IDF matrix<br/>36,801 x 79,041<br/>sparse"] -->|"TruncatedSVD(k=100)"| Z["LSA features<br/>36,801 x 100<br/>dense"]
    Z --> P["2-D plot,<br/>clustering (Session 11)"]
    Z --> M["Classifier or trees<br/>(compact features)"]
    X -->|"components_"| W["Word weights<br/>per component"]
```

### Why it matters

Truncated SVD gives a compact, dense representation that methods needing dense input can use: k-means clustering, nearest-neighbour search, plots, or the tree models of Session 10. On the case study it also shows what dominates the co-occurrence structure: the first components separate **languages** (French and Dutch function words against German ones), not product groups, because words of one language co-occur with each other. Only later components pick up topics such as footwear. 100 components keep 19 % of the variance, and a classifier on 300 LSA components reaches 0.615 accuracy against 0.772 on the full sparse matrix. LSA is therefore mainly a tool for exploration and for methods that cannot handle sparse input, and the historical step towards the learned embeddings of Session 14, which map texts of different languages into one space.

### How it works in Python

In [10]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

T = vec.transform(train["description"])
svd = TruncatedSVD(n_components=100, random_state=0).fit(T)
print(round(svd.explained_variance_ratio_.sum(), 2))      # 0.19 of the variance kept
for i in [1, 2, 3]:                                        # component 0 is a general average
    c = svd.components_[i]
    print(i, list(names[np.argsort(c)[::-1][:6]]), list(names[np.argsort(c)[:4]]))
# 1 ['de', 'en', 'la', 'et', 'un', 'une'] ['oberteil', 'aus', 'mit', 'laufsohlen']   French vs German
# 2 ['oberteil', 'laufsohlen', 'spinnstoff', 'kunststoff', 'schuhe', 'antrag'] ['die', 'es', 'um', 'handelt']
# 3 ['een', 'van', 'met', 'het', 'voor', 'volgende'] ['la', 'et', 'un', 'une']         Dutch vs French

# LSA features as classifier input: compact but much weaker than sparse TF-IDF (0.772)
lsa = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True),
                    TruncatedSVD(n_components=300, random_state=0), Normalizer(),
                    SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None,
                                  random_state=0, n_jobs=-1))
lsa.fit(train["description"], train["heading"])
pred_lsa = lsa.predict(valid["description"])
print(round(accuracy_score(valid["heading"], pred_lsa), 3),
      round(f1_score(valid["heading"], pred_lsa, average="macro"), 3))   # 0.615 0.296

0.19
1 ['de', 'en', 'la', 'et', 'un', 'une'] ['oberteil', 'aus', 'mit', 'laufsohlen']
2 ['oberteil', 'laufsohlen', 'spinnstoff', 'kunststoff', 'schuhe', 'antrag'] ['die', 'es', 'um', 'handelt']
3 ['een', 'van', 'met', 'het', 'voor', 'volgende'] ['la', 'et', 'un', 'une']


0.615 0.296


### In practice

- LSA was developed at Bellcore for information retrieval (Deerwester et al., 1990) so that a search for "car" could also find documents about "automobiles".
- Landauer and Dumais (1997) reported that LSA trained on an encyclopedia answered the synonym questions of the TOEFL English test about as well as non-native applicants to US universities.
- The scikit-learn example *Clustering text documents using k-means* ([extras notebook 04](extras/04-sklearn-document-clustering-lsa.ipynb)) uses LSA before k-means on news articles, because k-means works poorly on the raw sparse matrix.

> **Note.** The sign of an SVD component is arbitrary, and component 0 usually captures the average document (frequent words). Interpret components by their top *and* bottom words, and do not expect every component to have a clear meaning.

## The limits of word counts

### Concept

Bag-of-words, TF-IDF and n-grams share four blind spots, and the case study shows each of them:

1. **Other languages and synonyms.** "Spielzeugauto aus Kunststoff", "Voiture jouet en matière plastique" and "Toy car of plastics" describe the same good but share no word: their TF-IDF cosine similarity is 0. The model must learn each language separately from labelled examples; for Bulgarian or Slovak there are few.
2. **Word order beyond n words.** "Shoes with a textile upper and a leather sole" and "with a leather upper and a textile sole": bigrams help only if the material and the part stand next to each other.
3. **Ambiguity.** "Zubehör" (accessories) or "set" mean different things for different goods; one column has many meanings.
4. **Unknown words and short texts.** A word that is not in the training vocabulary gets no column. A short description gives the model little to go on: "Aufsteckbürste für elektrische Zahnbürsten" (replacement head for electric toothbrushes) is predicted as 6301 (blankets) instead of 9603 (brushes).

Each limit has a classic patch: n-grams (order), stemming and LSA (synonyms within one language), character n-grams (compounds, unknown words). None solves the problem in general, because all of them still represent a word by its identity, not by its meaning. **Language models** take the other route: they learn from large amounts of unlabelled text, in many languages, that "jouet", "Spielzeug" and "toy" occur in similar contexts, and represent texts as dense vectors in which such descriptions are close (Session 14).

```mermaid
timeline
    title From word counts to language models
    1972 : idf weighting (Spärck Jones)
    1990 : LSA, truncated SVD of the term matrix
    2003 : Topic models (LDA)
    2013 : word2vec word embeddings
    2017 : Transformer and attention
    2018 : BERT and GPT, pre-trained language models
    2019 : Multilingual sentence embeddings
```

### Why it matters

Knowing the limits explains part of the errors above (rare languages, short texts) and sets the expectations for Session 14: language models are not automatically better, but they address exactly these blind spots, in particular the language barrier. It also explains why TF-IDF remains a strong baseline: with 300,000 labelled decisions, every frequent heading has enough examples in the main languages, and TF-IDF models are fast, cheap and transparent.

### How it works in Python

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

same_good = ["Spielzeugauto aus Kunststoff", "Voiture jouet en matière plastique", "Toy car of plastics"]
V = TfidfVectorizer().fit_transform(same_good)
print(cosine_similarity(V).round(2))     # identity matrix: the three languages share no word
# [[1. 0. 0.]
#  [0. 1. 0.]
#  [0. 0. 1.]]

# a short description: the model sees only a few known words
print("zahnbürstenaufsatz" in vec.vocabulary_, "zahnbürste" in vec.vocabulary_)   # False True
guess = model.predict(["Aufsteckbürste für elektrische Zahnbürsten"])[0]
print(guess, nomenclature.loc[guess, "heading_description"][:30])   # 6301 Blankets and travelling rugs

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
False True
6301 Blankets and travelling rugs


### In practice

- Google reported in 2019 that applying the language model BERT to search queries helped mainly with longer queries where small words such as "to" and "for" change the meaning, a case that keyword matching handles poorly.
- Many production systems still use BM25 or TF-IDF for a first, cheap retrieval step and a language model only for the final ranking, combining the speed of counts with the semantics of embeddings (Session 15).
- Multilingual search in EU institutions relies on translation or cross-lingual representations because keyword matching cannot connect 24 languages; the same need arises when a German customs officer looks for similar French decisions.

> **Important.** Keep the TF-IDF classifier as the baseline for every later text model in the project. A language model is worth its extra cost only if it beats this baseline on the same validation years by a margin that matters (Session 14).

## Check your understanding

1. A model predicts 9503 for 40 decisions, 30 of them correctly; the validation set has 120 decisions of 9503. Compute precision, recall and F1 for 9503.
2. Always predicting 3926 gives accuracy 0.04 and macro-F1 0.0001. Why does macro-F1 punish this rule so much more than accuracy, and which of the two would a head of a classification unit care about first?
3. Why is a random train-validation split more optimistic than a split by year on this data set? Name two reasons.
4. Descriptions that quote their heading number are 98 % correct. Is this leakage? Compare it with the justification of the leakage section: when is each text written?
5. A teammate adds `keywords` to the description and reports 0.93 validation accuracy. What do you expect on the leaderboard, and how do you show it without submitting?

## Further reading

- Jurafsky, D. and Martin, J. H. (2026). *Speech and Language Processing*, 3rd ed. draft, chapters "Naive Bayes, Text Classification and Sentiment" and "Logistic Regression". https://web.stanford.edu/~jurafsky/slp3/
- scikit-learn developers. *Classification of text documents using sparse features* (example). https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html
- Joulin, A., Grave, E., Bojanowski, P. and Mikolov, T. (2017). Bag of tricks for efficient text classification. *Proceedings of EACL 2017*, 427–431. https://aclanthology.org/E17-2068/
- European Commission. *European Binding Tariff Information (EBTI)*: what a BTI decision is and how to search the database. https://taxation-customs.ec.europa.eu/online-services/online-services-and-databases-customs/european-binding-tariff-information-ebti_en
- Ng, A. (2018). *Machine Learning Yearning*, chapters 14–19 on error analysis. https://info.deeplearning.ai/machine-learning-yearning-book
- scikit-learn developers. *Clustering text documents using k-means* (example with LSA). https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html
- World Customs Organization. *General Interpretative Rules for the classification of goods in the Harmonized System*, part of the HS nomenclature overview. https://www.wcoomd.org/en/topics/nomenclature/overview/what-is-the-harmonized-system.aspx
- Ribeiro, M. T., Singh, S. and Guestrin, C. (2016). "Why should I trust you?": explaining the predictions of any classifier. *Proceedings of KDD 2016*, 1135–1144. https://arxiv.org/abs/1602.04938